In [ ]:
# ──────────────────────────────────────────────────────────────
# 사용자가 업로드한 이미지의 데이터를 바탕으로 질의응답 기능 담당 서버
# port 8000
# ──────────────────────────────────────────────────────────────

In [ ]:
# # ──────────────────────────────────────────────────────────────
# # 패키지 설치 (최초 1회)
## !pip install fastapi uvicorn ollama openai python-multipart python-dotenv
# # ──────────────────────────────────────────────────────────────
# import subprocess
# subprocess.run([
#     'pip', 'install', '-q',
#     'fastapi', 'uvicorn', 'ollama', 'openai',
#     'python-multipart', 'python-dotenv'
# ])

In [1]:

# ──────────────────────────────────────────────────────────────
# 임포트 및 .env 설정 로드
# ──────────────────────────────────────────────────────────────
import os
import io
# import base64
import threading
import uvicorn
from PIL import Image
import google.generativeai as genai
from dotenv import load_dotenv

from fastapi import FastAPI, File, Form, UploadFile, HTTPException
# from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

import ollama
from openai import OpenAI

load_dotenv(dotenv_path="..\..\dataset\config\.env")

d:\Git_Ddrive\ai_image_analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\SMT16\AppData\Local\Temp\ipykernel_23984\2398592616.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


True

In [2]:
# ──────────────────────────────────────────────────────────────
# 모델 호출 함수
# ──────────────────────────────────────────────────────────────

async def callOllama(imageBytes: bytes, question: str) -> str:
    """ Ollama 로컬 모델로 이미지 분석 후 답변 반환 """
    response = ollama.chat(
        model=os.getenv("OLLAMA_MODEL", 'gemma4:e2b'),
        messages=[
            {
                'role': 'user',
                'content': question,
                'images': [imageBytes]
            }
        ]
    )
    return response['message']['content']


async def callOpenAi(imageBytes: bytes, question: str) -> str:
    """ OpenAI models/gemini-1.5-flash로 이미지 분석 후 답변 반환 """
    # 1. 제미나이 설정
    api_key = os.getenv("OPENAI_API_KEY")
    genai.configure(api_key=api_key)

    # 지원되는 모델 리스트 가져오기 (generateContent가 가능한 모델만 필터링)
    available_models = [
        m.name for m in genai.list_models() 
        if 'generateContent' in m.supported_generation_methods
    ]

    # 2. 모델 결정
    target_model_name = os.getenv("OPENAI_MODEL", 'models/gemini-1.5-flash') 

    if target_model_name in available_models:
        final_model_name = target_model_name
        print(f"지정된 모델 사용: {final_model_name}")
    elif available_models:
        final_model_name = available_models[1]
        print(f"지정된 모델 '{target_model_name}'을 찾을 수 없어 첫 번째 모델 '{final_model_name}'을 사용합니다.")
    else:
        return "사용 가능한 Gemini 모델이 리스트에 없습니다. API 키나 권한을 확인하세요."

    model = genai.GenerativeModel(final_model_name)

    try:
        # 3. 바이트 데이터를 PIL 이미지 객체로 변환 (제미나이가 인식하는 형식)
        pil_image = Image.open(io.BytesIO(imageBytes))

        # 4. 제미나이 호출 (질문과 이미지를 리스트로 전달)
        # response.text가 바로 답변 내용입니다.
        response = model.generate_content([question, pil_image])


        # 5. 기존 return 형식에 맞춰 결과 텍스트만 반환
        return response.text, final_model_name

    except Exception as e:
        print(f"Gemini API Error: {str(e)}")
        return f"분석 중 오류가 발생했습니다: {str(e)}"

In [3]:
# ──────────────────────────────────────────────────────────────
# /analyze 엔드포인트
# ──────────────────────────────────────────────────────────────
app = FastAPI(title="Analyze Service")
USE_MODEL=os.getenv('USE_MODEL', "OLLAMA")

@app.post('/analyze')
async def analyzeImage(
    image: UploadFile = File(...),
    question: str = Form(default='이 이미지에서 텍스트를 추출하고 내용을 설명해줘.')
):
    """ 이미지와 질문을 받아 선택된 모델로 분석 결과를 반환하는 API """
    try:
        imageBytes = await image.read()

        if USE_MODEL == "OPEN_AI":
            answer, usedModel = await callOpenAi(imageBytes, question)
        elif USE_MODEL == "OLLAMA":
            answer = await callOllama(imageBytes, question)
            usedModel = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
        else:
            raise HTTPException(status_code=500, detail=f'알 수 없는 모델 설정: {USE_MODEL} {USE_MODEL == "OPEN_AI"}')

        return JSONResponse(content={
            'success': True,
            'model': usedModel,
            'question': question,
            'answer': answer
        })

    except HTTPException as httpErr:
        raise httpErr
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={'success': False, 'message': str(e)}
        )


@app.get('/')
async def healthCheck():
    """ 서버 상태 및 현재 모델 설정 확인 """
    return {'success': True, 'useModel': USE_MODEL}

In [ ]:
def run_server():
    # .env에서 포트를 가져오거나 기본값 8000 사용
    port = int(os.getenv("ANALYZE_PORT", 8000))
    # 노트북 충돌 방지를 위해 uvicorn.run 사용
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")

# 이미 서버가 돌고 있는지 확인 (재실행 시 에러 방지용)
# 주피터에서 셀을 여러 번 누를 때를 대비해 thread가 살아있는지 체크하면 좋습니다.
ocr_thread = threading.Thread(target=run_server, daemon=True)
ocr_thread.start()

print(f"🚀 analyze 서버가 백그라운드에서 시작되었습니다!")
print(f"문서 주소: http://localhost:{os.getenv('ANALYZE_PORT', 8000)}/docs")

🚀 analyze 서버가 백그라운드에서 시작되었습니다!
문서 주소: http://localhost:8000/docs


INFO:     Started server process [23984]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:51858 - "POST /analyze HTTP/1.1" 200 OK
